# 03 · Historical education → new establishments

The second leg of the mechanism: does 1990 schooling predict modern firm births?
Persistence is left out here so the coefficient is education's own association.
**Manuscript: Table 3, column (b).**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/cleaned_data_v3.csv")
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2010": "pop_2010",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989",
})

cols = ["Index_v2", "income_1989_real_2023", "pct_hs_1990",
        "pop_2010", "firms_2022", "firms_1989"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=["income_1989_real_2023", "pct_hs_1990",
                       "pop_2010", "firms_2022", "firms_1989", "State"])
df = df[(df["firms_2022"] > 0) & (df["pop_2010"] > 0) & (df["firms_1989"] > 0)]

df["log_firms_2022"] = np.log1p(df["firms_2022"])
df["log_pop_2010"] = np.log(df["pop_2010"])
df[["income_1989_scaled", "pct_hs_1990_scaled"]] = StandardScaler().fit_transform(
    df[["income_1989_real_2023", "pct_hs_1990"]])

# education + controls, no persistence
model = smf.ols(
    "log_firms_2022 ~ pct_hs_1990_scaled + income_1989_scaled "
    "+ log_pop_2010 + C(State)",
    data=df).fit(cov_type="HC3")
print(model.summary())